In [1]:
import sys

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns
import sys

# Add project root to python path

sys.path.append("../")

from src.config import RAW_DATA
from src.data_loader import DataLoader
from src.schema_validator import SchemaValidator
from src.preprocessing import Preprocessor
from src.visualization import Visualizer
from src.utils import *

plt.style.use("ggplot")

pd.set_option("display.max_columns",None)

In [2]:
loader = DataLoader(RAW_DATA)


In [3]:
data = loader.load_excel(
    "ethiopia_fi_unified_data.xlsx",
    sheet_name="data"
)

impact = loader.load_excel(
    "ethiopia_fi_unified_data.xlsx",
    sheet_name="impact_links"
)

reference = loader.load_excel(
    "reference_codes.xlsx"
)

print(data.shape)

print(impact.shape)

print(reference.shape)

DataLoadError: Worksheet named 'data' not found

In [ ]:
data.head()

In [ ]:
impact.head()

In [ ]:
impact.head()

In [ ]:
data.info()

In [ ]:
impact.info()

In [ ]:
reference.info()

In [ ]:
validator = SchemaValidator(data)

validator.run_all()

In [ ]:
missing_summary(data)

In [ ]:
validator.duplicate_records()

In [ ]:
record_counts = (

    data["record_type"]

    .value_counts()

    .reset_index()

)

record_counts.columns=[

    "Record Type",

    "Count"

]

record_counts

In [ ]:
plt.figure(figsize=(8,5))

sns.countplot(

    data=data,

    x="record_type"

)

plt.title("Record Type Distribution")

plt.show()

In [ ]:
pillar_counts = (

    data["pillar"]

    .value_counts()

)

pillar_counts

In [ ]:
plt.figure(figsize=(10,5))

sns.countplot(

    data=data,

    x="pillar"

)

plt.xticks(rotation=45)

plt.show()

In [ ]:
source_counts = (

    data["source_type"]

    .value_counts()

)

source_counts

In [ ]:
plt.figure(figsize=(12,5))

sns.countplot(

    data=data,

    x="source_type"

)

plt.xticks(rotation=45)

plt.show()

In [ ]:
obs = data[

    data.record_type=="observation"

]

obs.head()

In [ ]:
events = data[

    data.record_type=="event"

]

events.head()

In [ ]:
targets = data[

    data.record_type=="target"

]

targets.head()

In [ ]:
impact.head()

In [ ]:
obs["observation_date"] = pd.to_datetime(

    obs["observation_date"]

)

In [ ]:
print(

    obs["observation_date"].min()

)

print(

    obs["observation_date"].max()

)

In [ ]:
timeline = (

    obs

    .groupby(

        obs["observation_date"].dt.year

    )

    .size()

)

timeline

In [ ]:
timeline.plot(

    marker="o",

    figsize=(10,5)

)

plt.ylabel("Observations")

plt.title("Temporal Coverage")

plt.grid(True)

plt.show()

In [ ]:
indicator_counts=(

    obs["indicator_code"]

    .value_counts()

)

indicator_counts

In [ ]:
plt.figure(figsize=(15,6))

indicator_counts.plot.bar()

plt.xticks(rotation=90)

plt.show()

In [ ]:
sorted(

    obs["indicator_code"]

    .unique()

)

In [ ]:
events[[

    "indicator",

    "category",

    "observation_date"

]]

In [ ]:
events = events.sort_values(

    "observation_date"

)

events

In [ ]:
impact[[

    "parent_id",

    "pillar",

    "related_indicator",

    "impact_direction",

    "impact_magnitude",

    "lag_months"

]]

In [ ]:
merged = impact.merge(

    events,

    left_on="parent_id",

    right_on="record_id",

    suffixes=(

        "_impact",

        "_event"

    )

)

merged.head()

In [ ]:
pd.crosstab(

    merged["indicator"],

    merged["pillar_impact"]

)

In [ ]:
matrix = pd.crosstab(

    merged["indicator"],

    merged["pillar_impact"]

)

plt.figure(figsize=(12,7))

sns.heatmap(

    matrix,

    annot=True,

    cmap="Blues"

)

plt.title(

    "Event-Pillar Relationship"

)

plt.show()